In [1]:
from datasets import load_dataset
import pandas as pd
import re
from collections import defaultdict

In [2]:
mental_chat = load_dataset("ShenLab/MentalChat16K", split='train')
amod_counseling = load_dataset("Amod/mental_health_counseling_conversations", split='train')
cactus = load_dataset("LangAGI-Lab/cactus", split='train')
cbt_bench = load_dataset("Psychotherapy-LLM/CBT-Bench", "core_fine_seed", split="train")

In [3]:
def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'\\n', '\n', text)
    text = re.sub(r'\\t', '\t', text)
    text = text.replace('\\', '')
    text = re.sub(r'\s*\n\s*', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

In [4]:
advice_data = []

In [5]:
amod_grouped = defaultdict(list)

for each in amod_counseling:
    question = clean_text(each['Context'])
    response = clean_text(each['Response'])

    if question and response:
        amod_grouped[question].append(response)
    
for question, responses in amod_grouped.items():
    merge_responses = "\n".join([f"- Perspective {i+1}: {r}" for i, r in enumerate(responses[:3])])

    text_chunk = f"User Question: {question}\n\n### Support Advice: \n{merge_responses}"

    advice_data.append(
        {
            "text": text_chunk,
            "metadata": {
                "source_dataset": "amod_counseling",
                "source_type": "support_advice",
                "original_question": question[:100]
            }
        }
    )

for each in mental_chat:
    question = clean_text(str(each.get('input', '')))
    response = clean_text(str(each.get('output', '')))

    text_chunk = f"User Question: {question}\n\n### Support Advice:\n{response}"

    advice_data.append({
        "text": text_chunk,
        "metadata": {
            "source_dataset": "mental_chat",
            "source_type": "support_advice",
            "original_question": " ".join(question.split())[:100]
        }
    })

In [7]:
len(advice_data)

16979

In [8]:
advice_data[16970:16979]

[{'text': "User Question: I've been having a lot of problems in my relationships lately, and I'm hoping that through counseling, I can figure out why and how to improve them. Lately, it feels like every time I try to connect with someone, it just falls apart. I feel so alone and disconnected from the people around me.\nPossible reasons for these relationship issues could be my fear of abandonment and trust issues stemming from past experiences. These thoughts and emotions make it difficult for me to open up to others and form meaningful connections.\nFor example, recently I started dating someone and everything was going really well. But then, out of nowhere, I began feeling anxious and paranoid that they would leave me. This led to me becoming overly clingy and needy, which eventually pushed them away.\nThese symptoms of anxiety and fear of abandonment are persistent and have been affecting my relationships for a while now. Whenever a new relationship starts to become significant, I s

In [9]:
df_cactus = pd.DataFrame(cactus)

In [10]:
df_cactus.head()

,thought,patterns,intake_form,cbt_technique,cbt_plan,attitude,dialogue
0,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Decatastrophizing,Decatastrophizing\n\nCounseling plan:\n1. Iden...,negative,"Counselor: Good afternoon, Brooke. Thank you f..."
1,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Reality Testing,Reality Testing\n\nCounseling Plan:\n1. Identi...,neutral,"Counselor: Hello, Brooke. I'm glad you made it..."
2,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Reality Testing,Reality Testing\n\nCounseling Plan:\n1. Identi...,negative,"Counselor: Hello Brooke, I’m glad you could jo..."
3,I'm really into my crystals. People will proba...,"[labeling and mislabeling, jumping to conclusi...",Name:\nJonathan Emmett\nAge:\n37\nGender:\nmal...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,positive,"Counselor: Hello, Jonathan. It’s nice to meet ..."
4,I'm really into my crystals. People will proba...,"[labeling and mislabeling, jumping to conclusi...",Name:\nJonathan Emmett\nAge:\n37\nGender:\nmal...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,neutral,"Counselor: Hi Jonathan, I'm glad you could mak..."


In [11]:
df_cactus.dropna()

,thought,patterns,intake_form,cbt_technique,cbt_plan,attitude,dialogue
0,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Decatastrophizing,Decatastrophizing\n\nCounseling plan:\n1. Iden...,negative,"Counselor: Good afternoon, Brooke. Thank you f..."
1,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Reality Testing,Reality Testing\n\nCounseling Plan:\n1. Identi...,neutral,"Counselor: Hello, Brooke. I'm glad you made it..."
2,I frequent this animal shelter. All of the ani...,"[catastrophizing, discounting the positive, la...",Name:\nBrooke Davis\nAge:\n41\nGender:\nfemale...,Reality Testing,Reality Testing\n\nCounseling Plan:\n1. Identi...,negative,"Counselor: Hello Brooke, I’m glad you could jo..."
3,I'm really into my crystals. People will proba...,"[labeling and mislabeling, jumping to conclusi...",Name:\nJonathan Emmett\nAge:\n37\nGender:\nmal...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,positive,"Counselor: Hello, Jonathan. It’s nice to meet ..."
4,I'm really into my crystals. People will proba...,"[labeling and mislabeling, jumping to conclusi...",Name:\nJonathan Emmett\nAge:\n37\nGender:\nmal...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,neutral,"Counselor: Hi Jonathan, I'm glad you could mak..."
...,...,...,...,...,...,...,...
31572,I'll probably end up dying alone because I hav...,"[catastrophizing, discounting the positive, me...",Name:\nNathan Miller\nAge:\n36\nGender:\nmale\...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,positive,"Counselor: Hi, Nathan. I'm glad you could make..."
31573,I'll probably end up dying alone because I hav...,"[catastrophizing, discounting the positive, me...",Name:\nNathan Miller\nAge:\n36\nGender:\nmale\...,Alternative Perspective,Alternative Perspective\n\nCounseling Plan:\n1...,neutral,"Counselor: Hello Nathan, it's nice to meet you..."
31574,I'll probably end up dying alone because I hav...,"[catastrophizing, discounting the positive, me...",Name:\nNathan Miller\nAge:\n36\nGender:\nmale\...,Behavior Experiment,Behavior Experiment\n\nCounseling Plan:\n1. Id...,positive,"Counselor: Hi Nathan, it's nice to meet you. H..."
31575,I'll probably end up dying alone because I hav...,"[catastrophizing, discounting the positive, me...",Name:\nNathan Miller\nAge:\n36\nGender:\nmale\...,Behavior Experiment,Behavior Experiment\n\nCounseling Plan:\n1. Id...,negative,"Counselor: Hi Nathan, I'm glad you're here. Ho..."


In [12]:
df_cactus["patterns"] = df_cactus["patterns"].apply(lambda x: ", ".join(x))

In [13]:
df_cactus.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
31572    False
31573    False
31574    False
31575    False
31576    False
Length: 31577, dtype: bool

In [14]:
def extract_problem(text):
    if "2. Presenting Problem" in text:
        try:
            return text.split("2. Presenting Problem")[1].split("3. Reason for Seeking Counseling")[0].strip()
        except:
            return text[:300]
    return text

In [15]:
df_cactus['problem_cleaned'] = df_cactus['intake_form'].apply(extract_problem).apply(clean_text)
df_cactus['plan_cleaned'] = df_cactus['cbt_plan'].apply(clean_text)
df_cactus = df_cactus.drop_duplicates(subset=['problem_cleaned', 'cbt_technique'])

df_cactus['rag_text'] = (
    "SITUATION: " + df_cactus['problem_cleaned'] + "\n" +
    "NEGATIVE THOUGHT: " + df_cactus['thought'].apply(clean_text) + "\n" +
    "DISTORTIONS: " + df_cactus['patterns'].apply(clean_text) + "\n" +
    "TECHNIQUE: " + df_cactus['cbt_technique'].apply(clean_text) + "\n\n" +
    "### CBT EXERCISE PLAN:\n" + df_cactus['plan_cleaned']
)

In [16]:
for _, row in df_cactus.iterrows():
    advice_data.append({
        "text": row['rag_text'],
        "metadata": {
            "source_dataset": "cactus",
            "source_type": "cbt_template",
            "technique": row['cbt_technique']
        }
    })

In [20]:
len(advice_data)

28483

In [21]:
advice_data[28474:28483]

[{'text': 'SITUATION: I have been experiencing conflicting feelings about my job as a musician. Despite my job being great, I constantly dread going to work, leading to high levels of stress and unhappiness.\nThese feelings started around 6 months ago when I realized I was not finding joy in my passion for music as I used to.\nI believe the stress started due to a significant change in my band dynamics and increasing pressure to perform better.\nThe problem has worsened over time, with my dread of work increasing each day, even though I still love music.\nI experience these feelings every time I have a gig or band practice, and the pattern is consistent.\nI have tried to ignore these feelings and distract myself with other hobbies, but the dread persists.\nNEGATIVE THOUGHT: My job is great, but I hate it, I dread going to work.\nDISTORTIONS: discounting the positive, mental filtering, black-and-white or polarized thinking / all or nothing thinking, none\nTECHNIQUE: Alternative Perspect

In [22]:
df_bench = pd.DataFrame(cbt_bench)

In [23]:
df_bench

,id,ori_text,situation,thoughts,core_belief_fine_grained
0,46,From a teen in England: I lied to my boyfriend...,I lied to my boyfriend for over 1 1/2 years ab...,I feel guilty and ashamed for lying to my boyf...,"[I am unlovable, I am bound to be abandoned, I..."
1,4679,I realize that around my age and older is when...,I am extremely concerned that I may have schiz...,- I am out of control.\n- Something terrible w...,"[I am powerless, weak, vulnerable, I am needy,..."
2,35,From a young woman in Latvia: I should start b...,"I've never posted a question online before, as...","I feel trapped, I will never get new opportuni...","[I am helpless, I am powerless, weak, vulnerab..."
3,53,I was married 36 years to my soulmate. Our rel...,This client is struggling with unmet expectati...,"People should be reaching out to me more, Why ...","[I am a victim, I am bound to be rejected, I a..."
4,4674,My girlfriend is grieving over her ex-husband....,My girlfriend is grieving over her ex-husband....,- My girlfriend doesn't love me.\n- There is n...,"[I am helpless, I am powerless, weak, vulnerab..."
5,4623,I’m a student receiving my Masters degree. Wit...,I feel extremely anxious...I also feel pressur...,I'm having trouble focusing and making plans f...,"[I am helpless, I am out of control]"
6,54,"From a young woman in Algeria: overwhelmed, an...",This client is struggling with a lot of sympto...,"I am overwhelmed, I am stressed out, I cannot ...","[I am bound to be alone, I am worthless, waste..."
7,4690,It’s a mixture of family issues and an issue w...,"I'm 14, and live with my mother. When my mothe...",- I am out of control.\n- I have no say over m...,"[I am powerless, weak, vulnerable, I am out of..."
8,8,From a woman in the U.S.: My husband’s daughte...,The woman's husband's 19-year-old daughter fro...,I am being targeted.\nI am always mistreated.\...,"[I am a victim, I am trapped, I am a failure, ..."
9,4515,My husband and I have been together for almost...,"Ever since, I have known in the back of my min...",He calls that faithful? It doesn’t feel faithf...,"[I am trapped, I am out of control, I am defec..."


In [24]:
df_bench = pd.DataFrame(cbt_bench)

df_bench['sit_clean'] = df_bench['situation'].apply(clean_text)
df_bench['thought_clean'] = df_bench['thoughts'].apply(clean_text)

def format_beliefs(belief_item):
    if isinstance(belief_item, list):
        return ", ".join(belief_item)
    return str(belief_item)

df_bench['beliefs_clean'] = df_bench['core_belief_fine_grained'].apply(format_beliefs)

df_bench['rag_text'] = (
    "SITUATION: " + df_bench['sit_clean'] + "\n" +
    "AUTOMATIC THOUGHTS: " + df_bench['thought_clean'] + "\n\n" +
    "### UNDERLYING CORE BELIEFS:\n" +
    "This situation often links to deeper beliefs like: " + df_bench['beliefs_clean']
)

for _, row in df_bench.iterrows():
    advice_data.append({
        "text": row['rag_text'],
        "metadata": {
            "source_dataset": "cbt_bench_core",
            "source_type": "core_belief_discovery"
        }
    })

In [25]:
len(advice_data)

28503

In [26]:
advice_data[28500:28503]

[{'text': 'SITUATION: I can process a million things in my head at once but can’t seem to put them on paper. I am failing even though I am incredibly intelligent. I have a hard time sitting still. I mostly struggle with problem solving, not memorization.\nAUTOMATIC THOUGHTS: There is something wrong with me. I should not be failing because I am so smart. I will not be able to answer math questions.\n\n### UNDERLYING CORE BELIEFS:\nThis situation often links to deeper beliefs like: I am incompetent, I am a failure, loser, I am defective',
  'metadata': {'source_dataset': 'cbt_bench_core',
   'source_type': 'core_belief_discovery'}},
 {'text': "SITUATION: The problem in a nutshell is that I have no close friends and when I’m single (which is the case right now) I’m completely lonely and more depressed than ever...I’ve been so desperate to make friends that I’ve pushed myself to go to parties or events but when I get there I’m so awkward and introverted that I’m nearly brought to tears an

In [28]:
final_df = pd.DataFrame(advice_data)
final_df = final_df.drop_duplicates(subset=['text'])

print("--- Final Knowledge Base Stats ---")
print(final_df['metadata'].apply(lambda x: x['source_dataset']).value_counts())

final_df.to_json("clinical_kb.jsonl", orient="records", lines=True)

--- Final Knowledge Base Stats ---
metadata
mental_chat        16005
cactus             11504
amod_counseling      895
cbt_bench_core        20
Name: count, dtype: int64
